In [1]:
from pathlib import Path


def find_project_root(start: Path) -> Path:
    # Cerca la root del progetto risalendo dalle directory correnti.
    for candidate in [start, *start.parents]:
        if (candidate / 'agents').exists() and (candidate / 'data').exists() and (candidate / 'disciplines').exists():
            return candidate
    raise FileNotFoundError("Root progetto 'PhiloMind' non trovata.")


BASE = str(find_project_root(Path.cwd()))

Load principal data from : https://www.kaggle.com/datasets/kouroshalizadeh/history-of-philosophy/data

In [2]:
import pandas as pd
import seaborn as sns

df_train = pd.read_csv(f'{BASE}/data/raw/philosophy_data.csv')

ModuleNotFoundError: No module named 'pandas'

# Data exploration


In [ ]:
print(df_train.shape)
print(df_train.columns.tolist())
print(df_train['school'].value_counts())   # quante scuole filosofiche
print(df_train['author'].value_counts())   # quanti autori

print(df_train['school'].describe())
print(df_train['author'].describe())

sns.displot(df_train['school'])

### Exploring `sentence_length`

Visualizziamo la distribuzione delle lunghezze delle frasi per comprendere come sono generalmente strutturate nel dataset.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.histplot(df_train['sentence_length'], kde=True, bins=50)
plt.title('Distribution of Sentence Lengths')
plt.xlabel('Sentence Length')
plt.ylabel('Frequency')
plt.show()

### Exploring `original_publication_date`

Understanding the publication dates can show us the temporal distribution of the texts. We'll convert the column to datetime objects and then analyze the distribution of publication years.

In [ ]:
df_train['original_publication_date'] = pd.to_datetime(df_train['original_publication_date'], errors='coerce')

plt.figure(figsize=(12, 6))
sns.histplot(df_train['original_publication_date'].dt.year.dropna(), bins=30, kde=True)
plt.title('Distribution of Original Publication Years')
plt.xlabel('Publication Year')
plt.ylabel('Frequency')
plt.show()

### Checking for Missing Values

It's important to identify any missing values in the dataset, especially if we plan to use all features for modeling. Let's visualize the completeness of our data.

In [ ]:
import missingno as msno

# Visualize missing values
msno.matrix(df_train, figsize=(10, 5))
plt.title('Missing Values Matrix', fontsize=16)
plt.show()

# Display count of missing values per column
missing_counts = df_train.isnull().sum()
missing_percentages = (df_train.isnull().sum() / len(df_train)) * 100

print("\nMissing values count per column:")
print(missing_counts[missing_counts > 0])

print("\nMissing values percentage per column:")
print(missing_percentages[missing_percentages > 0])

### Relationship between Author and School

Let's investigate which authors are associated with which philosophical schools. A crosstab and heatmap can provide a clear visualization of these relationships.

In [ ]:
author_school_crosstab = pd.crosstab(df_train['author'], df_train['school'])

plt.figure(figsize=(15, 10))
sns.heatmap(author_school_crosstab > 0, cmap='Blues', cbar=False, linewidths=.5, linecolor='lightgray')
plt.title('Authors and their Associated Schools')
plt.xlabel('School')
plt.ylabel('Author')
plt.show()

### Sentence Length vs. School/Author

Now, let's explore if there's a difference in sentence length across different philosophical schools and authors. Box plots are suitable for this type of comparison.

In [ ]:
plt.figure(figsize=(15, 8))
sns.boxplot(x='school', y='sentence_length', data=df_train)
plt.title('Sentence Length Distribution by School')
plt.xlabel('School')
plt.ylabel('Sentence Length')
plt.xticks(rotation=45, ha='right')
plt.show()

plt.figure(figsize=(20, 10))
sns.boxplot(x='author', y='sentence_length', data=df_train)
plt.title('Sentence Length Distribution by Author')
plt.xlabel('Author')
plt.ylabel('Sentence Length')
plt.xticks(rotation=90, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
import sys
sys.path.append(BASE)

from agents.registry import AgentRegistry
print("=== CORPUS ===")
print(f"Brani totali: {len(df_train)}")
print(f"Autori: {df_train['author'].nunique()}")
print(f"Scuole: {df_train['school'].nunique()}")

print("\n=== DATASET CLASSIFICATORE ===")
labels = pd.read_csv(f'{BASE}/data/labels/questions_labeled.csv')
print(f"Domande totali: {len(labels)}")
print(labels['label'].value_counts())

print("\n=== REGISTRY ===")
registry = AgentRegistry(f'{BASE}/disciplines/config.json')
print(f"Discipline attive: {registry.list_disciplines()}")

### Data Normalization


In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Identify numerical columns for standardization
numerical_cols = ['sentence_length', 'original_publication_date_year', 'corpus_edition_date_year']

# Create a copy to avoid SettingWithCopyWarning
df_train_standardized = df_train.copy()

# Apply StandardScaler to the selected numerical columns
for col in numerical_cols:
    # Reshape the data for the scaler if it's a single column
    values = df_train_standardized[col].values.reshape(-1, 1)
    df_train_standardized[f'{col}_standardized'] = scaler.fit_transform(values)


print("DataFrame with standardized columns:")
display(df_train_standardized[['sentence_length', 'sentence_length_standardized',
                               'original_publication_date_year', 'original_publication_date_year_standardized',
                               'corpus_edition_date_year', 'corpus_edition_date_year_standardized']].head())